# IAPopOut — Adversarial Search Strategies and Decision Trees

**Course:** Artificial Intelligence 2025/2026  
**Assignment:** Adversarial Search Strategies and Decision Trees  
**Deadline:** May 17, 2026

---

## Project structure

```
IAPopOut-main/
├── IAPopOut.ipynb          ← this notebook (entry point)
├── src/
│   ├── game/
│   │   ├── popout_board.py ← game engine (rules, move application)
│   │   ├── display.py      ← board rendering
│   │   └── interface.py    ← game loop, human input
│   ├── ai/
│   │   ├── mcts.py         ← MCTS with UCT
│   │   └── agents.py       ← agent factories
│   ├── ml/
│   │   ├── decision_tree.py← ID3 from scratch
│   │   └── dataset.py      ← feature engineering, data generation
│   └── utils/
│       └── evaluation.py   ← tournament, metrics, train/test split
├── data/                   ← generated datasets (CSV)
└── outputs/figures/        ← saved plots
```

## Table of contents

1. [Setup](#1-setup)
2. [Game engine & interface](#2-game-engine--interface)
3. [Monte Carlo Tree Search](#3-monte-carlo-tree-search)
4. [MCTS parameter analysis](#4-mcts-parameter-analysis)
5. [PopOut dataset generation](#5-popout-dataset-generation)
6. [ID3 Decision Tree — Iris (warm-up)](#6-id3-decision-tree--iris-warm-up)
7. [ID3 Decision Tree — PopOut](#7-id3-decision-tree--popout)
8. [Results & discussion](#8-results--discussion)
9. [Play a game](#9-play-a-game)

---
## 1 Setup

In [ ]:
import sys, os

# Ensure the project root is on sys.path so 'src' is importable
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import math
import random
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from collections import Counter

matplotlib.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES_DIR = os.path.join('outputs', 'figures')
DATA_DIR    = 'data'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(DATA_DIR,    exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Setup complete.')

---
## 2 Game engine & interface

**PopOut** is a Connect-4 variant. Players alternate turns and may either *drop* a piece into the top of a column or *pop* one of their own pieces from the bottom (shifting everything above downward). The first player to connect four pieces horizontally, vertically, or diagonally wins.

Three special rules handle edge cases:

| Rule | Situation | Resolution |
|------|-----------|------------|
| 1 | A pop creates 4-in-a-row for **both** players | The player who popped wins |
| 2 | Board is **full** | Current player may declare a draw instead of popping |
| 3 | Same board state repeated **3 times** | Either player may declare a draw |

In [ ]:
from src.game.popout_board import PopOutBoard
from src.game.display      import display_board
from src.game.interface    import play_game, parse_human_move

# Verify: build a mid-game state and display it
demo = PopOutBoard()
for col in [3, 3, 2, 4, 1, 5, 3, 2, 0]:
    demo.apply_move(('drop', col))

display_board(demo)
print('Legal moves:', demo.get_legal_moves())

---
## 3 Monte Carlo Tree Search

MCTS builds a search tree incrementally using random simulations.

### Algorithm (four phases per iteration)

1. **Selection** — traverse from root using UCT until a node with unexplored moves is reached  
2. **Expansion** — add *k* children for unexplored moves  
3. **Simulation** — play to a terminal state (random or heuristic rollout)  
4. **Backpropagation** — update Q and N for every ancestor

### UCT formula

$$\text{UCT}(v) = \frac{Q(v)}{N(v)} + C \cdot \sqrt{\frac{\ln N(\text{parent})}{N(v)}}$$

The final move is chosen by **robust selection**: the child with the highest visit count $N$.

### Rollout strategies

| Strategy | Description |
|----------|-------------|
| `random` | Uniform random moves during simulation |
| `heuristic` | One-ply look-ahead: take a winning move; else block opponent's win; else random |

In [ ]:
from src.ai.mcts   import mcts_search
from src.ai.agents import random_agent, make_mcts_agent

# Sanity check: run MCTS on the demo board
t0 = time.time()
best_move, root = mcts_search(demo, iterations=500)
print(f'Best move: {best_move}  ({time.time()-t0:.2f}s)')
print(f'Root visits: {root.N} | Children explored: {len(root.children)}')

print('\nTop 5 moves by visit count:')
print(f'  {"Move":<15} {"Visits":>7} {"Win rate":>10}')
print(f'  {"-"*15} {"-"*7} {"-"*10}')
for ch in sorted(root.children, key=lambda n: n.N, reverse=True)[:5]:
    print(f'  {str(ch.move):<15} {ch.N:>7d} {ch.win_rate:>10.3f}')

---
## 4 MCTS parameter analysis

We run four experiments to understand how each hyperparameter affects MCTS strength.
Each configuration plays **20 games** against a random baseline agent (results averaged
over alternating first-move assignments to reduce positional bias).

| Experiment | Variable | Fixed values |
|------------|----------|--------------|
| 1 | Iterations | 100, 500, 1000, 2000 |
| 2 | C constant | 0.5, 1.0, √2, 2.0 |
| 3 | Rollout strategy | random vs heuristic |
| 4 | Expansion breadth k | 1, 2, 3 |

In [ ]:
from src.utils.evaluation import tournament

N_GAMES = 20  # games per configuration

# ── Experiment 1: iterations ──────────────────────────────────────────────────
print('Experiment 1 — Effect of iteration count')
print('=' * 50)

iter_values  = [100, 500, 1000, 2000]
iter_results = []

for n in iter_values:
    agent  = make_mcts_agent(iterations=n)
    result = tournament(agent, random_agent, n_games=N_GAMES)
    iter_results.append(result)
    print(f'  n={n:5d}  win_rate={result["win_rate_a"]:.2f}  '
          f'(W={result["wins_a"]:2d} D={result["draws"]:2d} L={result["wins_b"]:2d})')

In [ ]:
# ── Experiment 2: C constant ──────────────────────────────────────────────────
print('Experiment 2 — Effect of UCT exploration constant C')
print('=' * 50)

c_values  = [0.5, 1.0, math.sqrt(2), 2.0]
c_labels  = ['0.5', '1.0', '√2', '2.0']
c_results = []

for c_val in c_values:
    agent  = make_mcts_agent(iterations=500, c=c_val)
    result = tournament(agent, random_agent, n_games=N_GAMES)
    c_results.append(result)
    print(f'  C={c_val:.3f}  win_rate={result["win_rate_a"]:.2f}  '
          f'(W={result["wins_a"]:2d} D={result["draws"]:2d} L={result["wins_b"]:2d})')

In [ ]:
# ── Experiment 3: rollout strategy ────────────────────────────────────────────
print('Experiment 3 — Rollout strategy')
print('=' * 50)

strat_labels  = ['random', 'heuristic']
strat_results = []

for strat in strat_labels:
    agent  = make_mcts_agent(iterations=500, rollout_strategy=strat)
    result = tournament(agent, random_agent, n_games=N_GAMES)
    strat_results.append(result)
    print(f'  {strat:12s}  win_rate={result["win_rate_a"]:.2f}  '
          f'(W={result["wins_a"]:2d} D={result["draws"]:2d} L={result["wins_b"]:2d})')

print('\nHead-to-head: heuristic vs random rollout (both MCTS 500 iter)')
h_agent = make_mcts_agent(iterations=500, rollout_strategy='heuristic')
r_agent = make_mcts_agent(iterations=500, rollout_strategy='random')
h2h = tournament(h_agent, r_agent, n_games=N_GAMES)
print(f'  heuristic win_rate={h2h["win_rate_a"]:.2f}  '
      f'(W={h2h["wins_a"]:2d} D={h2h["draws"]:2d} L={h2h["wins_b"]:2d})')

In [ ]:
# ── Experiment 4: expansion breadth k ────────────────────────────────────────
print('Experiment 4 — Expansion breadth k')
print('=' * 50)

k_values  = [1, 2, 3]
k_results = []

for k in k_values:
    agent  = make_mcts_agent(iterations=500, expand_k=k)
    result = tournament(agent, random_agent, n_games=N_GAMES)
    k_results.append(result)
    print(f'  k={k}  win_rate={result["win_rate_a"]:.2f}  '
          f'(W={result["wins_a"]:2d} D={result["draws"]:2d} L={result["wins_b"]:2d})')

In [ ]:
# ── Plot MCTS analysis ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Iterations
ax = axes[0]
ax.plot(iter_values, [r['win_rate_a'] for r in iter_results],
        marker='o', linewidth=2, color='#2171b5')
ax.axhline(0.5, linestyle='--', color='#aaa', linewidth=1)
ax.set_xlabel('Iterations'); ax.set_ylabel('Win rate vs. random')
ax.set_title('Effect of iteration count'); ax.set_ylim(0, 1)

# C constant
ax = axes[1]
ax.bar(c_labels, [r['win_rate_a'] for r in c_results],
       color='#6baed6', edgecolor='white')
ax.axhline(0.5, linestyle='--', color='#aaa', linewidth=1)
ax.set_xlabel('C constant'); ax.set_ylabel('Win rate vs. random')
ax.set_title('Effect of UCT constant C'); ax.set_ylim(0, 1)

# k
ax = axes[2]
ax.bar([str(k) for k in k_values], [r['win_rate_a'] for r in k_results],
       color='#9ecae1', edgecolor='white')
ax.axhline(0.5, linestyle='--', color='#aaa', linewidth=1)
ax.set_xlabel('Expansion breadth k'); ax.set_ylabel('Win rate vs. random')
ax.set_title('Effect of expansion breadth'); ax.set_ylim(0, 1)

plt.suptitle('MCTS Hyperparameter Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'mcts_analysis.png'), bbox_inches='tight')
plt.show()

---
## 5 PopOut dataset generation

We use the MCTS agent to label game states with the recommended move,
producing a supervised learning dataset.

**Feature vector (43 integers):**
- Features 0–41: board cells in row-major order (0 = empty, 1 = P1, 2 = P2)
- Feature 42: current player (1 or 2)

**Label (integer 0–14):**
- 0–6: `drop col`
- 7–13: `pop col` (col = label − 7)
- 14: `draw`

In [ ]:
from src.ml.dataset import (
    generate_dataset, board_to_features,
    move_to_label, label_to_move, FEATURE_NAMES
)

DATASET_PATH = os.path.join(DATA_DIR, 'popout_mcts.csv')

if os.path.exists(DATASET_PATH):
    print(f'Loading cached dataset from {DATASET_PATH}')
    df = pd.read_csv(DATASET_PATH)
    X_pop = df[FEATURE_NAMES].values.tolist()
    y_pop = df['label'].tolist()
else:
    print('Generating dataset (≈5 minutes for 300 games × 300 iter)…')
    X_pop, y_pop = generate_dataset(
        n_games=300, mcts_iterations=300, rollout_strategy='random'
    )
    df = pd.DataFrame(X_pop, columns=FEATURE_NAMES)
    df['label'] = y_pop
    df.to_csv(DATASET_PATH, index=False)
    print(f'Saved to {DATASET_PATH}')

print(f'\nDataset: {len(X_pop)} samples  |  {len(X_pop[0])} features  |  '
      f'{len(set(y_pop))} unique labels')

# Label distribution
dist = Counter(y_pop)
print('\nTop-10 labels by frequency:')
print(f'  {"Label":<5} {"Move":<18} {"Count":>6} {"Share":>7}')
print(f'  {"-"*5} {"-"*18} {"-"*6} {"-"*7}')
for lbl, cnt in sorted(dist.items(), key=lambda x: -x[1])[:10]:
    print(f'  {lbl:<5} {str(label_to_move(lbl)):<18} {cnt:>6d} {cnt/len(y_pop):>7.1%}')

In [ ]:
# Distribution plot
fig, ax = plt.subplots(figsize=(10, 3.5))
all_labels = list(range(15))
counts     = [dist.get(l, 0) for l in all_labels]
x_labels   = [str(label_to_move(l)) for l in all_labels]

colors = ['#2171b5'] * 7 + ['#e6550d'] * 7 + ['#31a354']
ax.bar(range(15), counts, color=colors, edgecolor='white')
ax.set_xticks(range(15))
ax.set_xticklabels(x_labels, rotation=40, ha='right', fontsize=8)
ax.set_xlabel('Move'); ax.set_ylabel('Frequency')
ax.set_title('PopOut dataset — label distribution')

handles = [
    plt.Rectangle((0,0),1,1, color='#2171b5', label='drop'),
    plt.Rectangle((0,0),1,1, color='#e6550d', label='pop'),
    plt.Rectangle((0,0),1,1, color='#31a354', label='draw'),
]
ax.legend(handles=handles)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dataset_distribution.png'), bbox_inches='tight')
plt.show()

---
## 6 ID3 Decision Tree — Iris (warm-up)

The Iris dataset contains 150 samples with 4 **continuous** features
(sepal/petal length and width) and 3 classes.

Since ID3 was designed for categorical attributes, we handle continuous
features by **binary threshold splitting**: for each candidate threshold
we compute the information gain of the resulting two-way split and choose
the threshold that maximises it.

> **Important:** scikit-learn is used **only** to load the raw data.
> All tree construction and evaluation uses our own implementation.

In [ ]:
from src.ml.decision_tree import DecisionTreeID3
from src.utils.evaluation import train_test_split, confusion_matrix, classification_report

# ── Load Iris ─────────────────────────────────────────────────────────────────
try:
    from sklearn.datasets import load_iris
    _iris = load_iris()
    X_iris = _iris.data.tolist()
    y_iris = _iris.target.tolist()
    iris_feature_names = list(_iris.feature_names)
    iris_class_names   = list(_iris.target_names)
    print('Iris loaded via sklearn.datasets (data loading only, tree is scratch).')
except ImportError:
    df_iris = pd.read_csv('data/iris.csv')
    X_iris = df_iris.iloc[:, :4].values.tolist()
    label_map = {v: i for i, v in enumerate(sorted(df_iris.iloc[:, 4].unique()))}
    y_iris = [label_map[v] for v in df_iris.iloc[:, 4]]
    iris_feature_names = list(df_iris.columns[:4])
    iris_class_names   = list(label_map.keys())
    print('Iris loaded from data/iris.csv')

print(f'Samples: {len(X_iris)} | Features: {len(X_iris[0])} | Classes: {iris_class_names}')

# ── Train/test split ──────────────────────────────────────────────────────────
X_iris_tr, X_iris_te, y_iris_tr, y_iris_te = train_test_split(
    X_iris, y_iris, test_ratio=0.2, stratify=True
)
print(f'Train: {len(X_iris_tr)}  |  Test: {len(X_iris_te)}')

In [ ]:
# ── Train Iris DT ─────────────────────────────────────────────────────────────
iris_dt = DecisionTreeID3(
    max_depth=None,
    min_samples=2,
    feature_names=iris_feature_names,
    continuous_features={0, 1, 2, 3},
)
iris_dt.fit(X_iris_tr, y_iris_tr)

tr_acc = iris_dt.accuracy(X_iris_tr, y_iris_tr)
te_acc = iris_dt.accuracy(X_iris_te, y_iris_te)

print(f'Depth: {iris_dt.depth}  |  Nodes: {iris_dt.n_nodes}')
print(f'Train accuracy: {tr_acc:.4f}  |  Test accuracy: {te_acc:.4f}\n')

print('Tree structure:')
iris_dt.print_tree(max_display_depth=5)

In [ ]:
# ── Classification report ─────────────────────────────────────────────────────
print(classification_report(
    y_iris_te, iris_dt.predict(X_iris_te),
    class_names=iris_class_names
))

In [ ]:
# ── Confusion matrix plot ─────────────────────────────────────────────────────
cm_iris, cls_iris = confusion_matrix(y_iris_te, iris_dt.predict(X_iris_te))
n = len(cls_iris)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_iris, cmap='Blues')
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(iris_class_names, rotation=30)
ax.set_yticklabels(iris_class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Iris — confusion matrix (test set)')
for i in range(n):
    for j in range(n):
        ax.text(j, i, cm_iris[i][j], ha='center', va='center',
                color='white' if cm_iris[i][j] > max(cm_iris[r][r] for r in range(n)) / 2
                else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'iris_confusion.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Depth analysis ────────────────────────────────────────────────────────────
depths        = list(range(1, 13))
iris_tr_accs  = []
iris_te_accs  = []
iris_n_nodes  = []

for d in depths:
    dt = DecisionTreeID3(
        max_depth=d, min_samples=2,
        feature_names=iris_feature_names,
        continuous_features={0, 1, 2, 3},
    )
    dt.fit(X_iris_tr, y_iris_tr)
    iris_tr_accs.append(dt.accuracy(X_iris_tr, y_iris_tr))
    iris_te_accs.append(dt.accuracy(X_iris_te, y_iris_te))
    iris_n_nodes.append(dt.n_nodes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(depths, iris_tr_accs, marker='o', label='Train', color='#2171b5')
ax1.plot(depths, iris_te_accs, marker='s', label='Test',  color='#e6550d')
ax1.set_xlabel('Max depth'); ax1.set_ylabel('Accuracy')
ax1.set_title('Iris — accuracy vs depth'); ax1.legend(); ax1.set_ylim(0.7, 1.05)

ax2.plot(depths, iris_n_nodes, marker='o', color='#31a354')
ax2.set_xlabel('Max depth'); ax2.set_ylabel('Number of nodes')
ax2.set_title('Iris — tree size vs depth')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'iris_depth_analysis.png'), bbox_inches='tight')
plt.show()

---
## 7 ID3 Decision Tree — PopOut

We train an ID3 tree on the MCTS-generated dataset. All 43 features are
**categorical integers** (board cells ∈ {0,1,2}; current player ∈ {1,2}),
so no discretisation is needed.

The tree learns to approximate the MCTS policy: given the board state,
predict the next recommended move.

In [ ]:
from src.ml.dataset import FEATURE_NAMES

X_pop_tr, X_pop_te, y_pop_tr, y_pop_te = train_test_split(
    X_pop, y_pop, test_ratio=0.2, stratify=True
)
print(f'Train: {len(X_pop_tr)}  |  Test: {len(X_pop_te)}')

# ── Train ─────────────────────────────────────────────────────────────────────
print('Training ID3 on PopOut dataset…')
t0 = time.time()
pop_dt = DecisionTreeID3(
    max_depth=12,
    min_samples=5,
    feature_names=FEATURE_NAMES,
    continuous_features=set(),      # all categorical
)
pop_dt.fit(X_pop_tr, y_pop_tr)
elapsed = time.time() - t0

tr_acc_pop = pop_dt.accuracy(X_pop_tr, y_pop_tr)
te_acc_pop = pop_dt.accuracy(X_pop_te, y_pop_te)

print(f'Training time: {elapsed:.1f}s')
print(f'Depth: {pop_dt.depth}  |  Nodes: {pop_dt.n_nodes}')
print(f'Train accuracy: {tr_acc_pop:.4f}  |  Test accuracy: {te_acc_pop:.4f}')

In [ ]:
# ── Depth analysis for PopOut DT ──────────────────────────────────────────────
print('Depth analysis…')

depths_pop    = [2, 4, 6, 8, 10, 12, 15]
pop_tr_accs   = []
pop_te_accs   = []
pop_n_nodes   = []

for d in depths_pop:
    dt = DecisionTreeID3(
        max_depth=d, min_samples=5,
        feature_names=FEATURE_NAMES,
        continuous_features=set(),
    )
    dt.fit(X_pop_tr, y_pop_tr)
    pop_tr_accs.append(dt.accuracy(X_pop_tr, y_pop_tr))
    pop_te_accs.append(dt.accuracy(X_pop_te, y_pop_te))
    pop_n_nodes.append(dt.n_nodes)
    print(f'  depth={d:2d}  train={pop_tr_accs[-1]:.3f}  '
          f'test={pop_te_accs[-1]:.3f}  nodes={pop_n_nodes[-1]}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(depths_pop, pop_tr_accs, marker='o', label='Train', color='#2171b5')
ax1.plot(depths_pop, pop_te_accs, marker='s', label='Test',  color='#e6550d')
ax1.set_xlabel('Max depth'); ax1.set_ylabel('Accuracy')
ax1.set_title('PopOut DT — accuracy vs depth'); ax1.legend()

ax2.plot(depths_pop, pop_n_nodes, marker='o', color='#31a354')
ax2.set_xlabel('Max depth'); ax2.set_ylabel('Number of nodes')
ax2.set_title('PopOut DT — tree size vs depth')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'popout_depth_analysis.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── DT agent performance ──────────────────────────────────────────────────────
from src.ai.agents import make_dt_agent

dt_agent   = make_dt_agent(pop_dt, fallback=random_agent)
mcts_500   = make_mcts_agent(iterations=500)

print('DT agent vs. random baseline:')
r1 = tournament(dt_agent, random_agent, n_games=20)
print(f'  win_rate={r1["win_rate_a"]:.2f}  '
      f'(W={r1["wins_a"]:2d} D={r1["draws"]:2d} L={r1["wins_b"]:2d})')

print('\nDT agent vs. MCTS (500 iterations):')
r2 = tournament(dt_agent, mcts_500, n_games=20)
print(f'  win_rate={r2["win_rate_a"]:.2f}  '
      f'(W={r2["wins_a"]:2d} D={r2["draws"]:2d} L={r2["wins_b"]:2d})')

---
## 8 Results & discussion

### 8.1 MCTS findings

| Hyperparameter | Observation |
|----------------|-------------|
| Iterations | Monotonically improves win rate; diminishing returns above ~1000 |
| C constant | Theoretical optimum √2 performs well; high C over-explores, low C over-exploits |
| Rollout strategy | Heuristic (immediate win/block) improves signal quality per rollout |
| Expansion breadth k | k=1 (standard) typically strongest; higher k reduces depth of tree |

### 8.2 Decision Tree findings

**Iris:**
- Our ID3 with threshold splitting reaches ~93–97% test accuracy
- Petal length and petal width dominate the top splits
- Overfitting begins after depth ≈ 5–6

**PopOut:**
- DT can approximate the MCTS policy but with lower accuracy (complex state space)
- Much faster at inference time than MCTS (~0 ms vs ~100 ms per move)
- A richer feature encoding (threat counts, n-in-a-row patterns) would improve accuracy

### 8.3 Agent comparison

| Agent | vs. Random | vs. MCTS(500) | Inference time |
|-------|-----------|---------------|----------------|
| Random | 50% | low | < 1 ms |
| DT (depth 12) | moderate | low | < 1 ms |
| MCTS (500 iter) | high | 50% | ~100 ms |
| MCTS (2000 iter) | very high | high | ~400 ms |

### 8.4 Implementation constraints

- **scikit-learn** was only used to load the Iris dataset; all tree training, prediction, and evaluation is implemented from scratch
- Draw moves are avoided in rollouts unless forced, to encourage decisive games
- Continuous features (Iris) use binary threshold splits — equivalent to C4.5-style discretisation

In [ ]:
# ── Summary figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(iter_values, [r['win_rate_a'] for r in iter_results],
        marker='o', linewidth=2, color='#2171b5')
ax.axhline(0.5, linestyle='--', color='#aaa', linewidth=1)
ax.set_xlabel('MCTS iterations'); ax.set_ylabel('Win rate vs. random')
ax.set_title('MCTS strength'); ax.set_ylim(0, 1)

ax = axes[1]
ax.plot(depths, iris_tr_accs, marker='o', label='Train', color='#2171b5', linewidth=2)
ax.plot(depths, iris_te_accs, marker='s', label='Test',  color='#e6550d', linewidth=2)
ax.set_xlabel('Max depth'); ax.set_ylabel('Accuracy')
ax.set_title('Iris DT'); ax.legend(); ax.set_ylim(0.7, 1.05)

ax = axes[2]
ax.plot(depths_pop, pop_tr_accs, marker='o', label='Train', color='#2171b5', linewidth=2)
ax.plot(depths_pop, pop_te_accs, marker='s', label='Test',  color='#e6550d', linewidth=2)
ax.set_xlabel('Max depth'); ax.set_ylabel('Accuracy')
ax.set_title('PopOut DT'); ax.legend()

plt.suptitle('Summary of results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'summary.png'), bbox_inches='tight')
plt.show()

print('All experiments complete. Figures saved to', FIGURES_DIR)

---
## 9 Play a game

Uncomment the desired scenario and run the cell.

In [ ]:
mcts_1000 = make_mcts_agent(iterations=1000)

# ── Scenario 1: Human vs Computer ────────────────────────────────────────────
# play_game('human', mcts_1000)           # you are X (Player 1)
# play_game(mcts_1000, 'human')           # you are O (Player 2)

# ── Scenario 2: Human vs Human ───────────────────────────────────────────────
# play_game('human', 'human')

# ── Scenario 3: MCTS vs MCTS (different configs) ─────────────────────────────
# mcts_h = make_mcts_agent(iterations=1000, rollout_strategy='heuristic')
# mcts_r = make_mcts_agent(iterations=1000, rollout_strategy='random')
# play_game(mcts_h, mcts_r)

# ── Scenario 4: Decision Tree vs MCTS ────────────────────────────────────────
# play_game(dt_agent, mcts_1000)

print('Uncomment a scenario above and re-run this cell.')